# Merging new SOC results
This notebook is to merge all new results generated using the new FAO GAEZ yields as inputs, including reduced tillage and just crops.

## SETUP

### Modules

In [107]:
import pandas as pd
import polars as pl
import geopandas as gpd
import sbtn_leaf.map_plotting as mp
import sbtn_leaf.paths as sbtn_path
import sqlite3

LEAFS_FOLDER = sbtn_path._LEAFS_DIR

### Data

Data paths

In [25]:
# Geopakcages
gpckg_og_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_country.gpkg"
gpckg_og_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry.gpkg"
gpckg_og_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion.gpkg"

gpckg_gaez_ctr_path   = LEAFS_FOLDER / "SOC_2030_country_crops_clipped.gpkg"
gpckg_gaez_sc_path    = LEAFS_FOLDER / "SOC_2030_subcountry_crops_clipped.gpkg"
gpckg_gaez_er_path    = LEAFS_FOLDER / "SOC_2030_ecoregions_crops_clipped.gpkg"

gpckg_rt_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.gpkg"
gpckg_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.gpkg"
gpckg_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.gpkg"

#gpcks layers names
ctr_layer = "soc_leaf_country"
sc_layer = "soc_leaf_subcountry"
er_layer = "soc_leaf_ecoregions"
geom_layer = "geometry_layer"

# csvs
csv_og_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_country.csv"
csv_og_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry.csv"
csv_og_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion.csv"

csv_gaez_ctr_path   = LEAFS_FOLDER / "SOC_2030_country_crops_clipped.csv"
csv_gaez_sc_path    = LEAFS_FOLDER / "SOC_2030_subcountry_crops_clipped.csv"
csv_gaez_er_path    = LEAFS_FOLDER / "SOC_2030_ecoregions_crops_clipped.csv"

csv_rt_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.csv"
csv_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.csv"
csv_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.csv"

Opening geopackages

In [47]:
og_country_gpckg = gpd.read_file(gpckg_og_ctr_path, layer=geom_layer)
og_country_df = pd.read_csv(csv_og_ctr_path)
og_country_gdf = og_country_gpckg.merge(og_country_df.drop(columns="country"), how="left", on="ADM0_NAME")

In [48]:
gaez_country_gpckg = gpd.read_file(gpckg_gaez_ctr_path, layer=ctr_layer)
gaez_country_geom =  gpd.read_file(gpckg_gaez_ctr_path, layer=geom_layer)
gaez_country_gdf = gaez_country_geom.merge(gaez_country_gpckg.drop(columns="_source_file"), how="left", on="ADM0_NAME")

In [49]:
rt_country_gpckg = gpd.read_file(gpckg_rt_ctr_path, layer=ctr_layer)
rt_country_geom =  gpd.read_file(gpckg_rt_ctr_path, layer=geom_layer)
rt_country_gdf = rt_country_geom.merge(rt_country_gpckg.drop(columns="_source_file"), how="left", on="ADM0_NAME")

## Merging
So there are 3 things to be done in the merger: 1) Detect all crops flows, 2) Exchange those crop flows for the new GAEZ ones, and 3) Add the reduced tillage ones.

### Countries

All the crops present in the GAEZ files needs to be exchanged, so a list will be created, then eliminated from the original geopackage, and then replaced by the GAEZ ones.

In [88]:
gaez_country_flows_list = gaez_country_gpckg["flow_name"].unique()

In [89]:
non_crops_og_country_df = og_country_df[~og_country_df["flow_name"].isin(gaez_country_flows_list)].drop(columns = "country")

So there are a couples of things that still need to be filtered out, including West Bank results, Maize, rainfed. 

In [90]:
non_crops_og_country_df = non_crops_og_country_df[(non_crops_og_country_df["ADM0_NAME"] != "West Bank") & (non_crops_og_country_df["flow_name"]!= "Maize_rf_2030y_SOC")] 

Now, because of some weird reason the geopackage of the country leaf layer doesn't exist, so need to work with the csv, which is in long format, which needs to be transformed into wide format.

In [91]:
non_crops_og_country_df_wide = non_crops_og_country_df.pivot(values = "value", columns="metric", index=["ADM0_NAME", "flow_name"]).reset_index().rename(columns={"cf_mean": "cf"})

Dropping the _source_file column of the other 2 geopackages

In [ ]:
gaez_country_gpckg = gaez_country_gpckg.drop(columns = "_source_file")
rt_country_gpckg = rt_country_gpckg.drop(columns = "_source_file")

Stacking everything

In [101]:
final_country_df = pd.concat([non_crops_og_country_df_wide, gaez_country_gpckg, rt_country_gpckg], ignore_index= True)

Finally saving the file

In [110]:
country_final_gpckg_path = LEAFS_FOLDER / "SOC/SOC_2030_country_v1.0.gpkg"

In [ ]:
og_country_gpckg.to_file(country_final_gpckg_path, layer = geom_layer, driver="GPKG")

In [111]:
# Write values table directly into the geopackage (which is just a sqlite db)
conn = sqlite3.connect(country_final_gpckg_path)
final_country_df.to_sql(ctr_layer, conn, if_exists="replace", index=False)
conn.close()

In [108]:
final_country_df.columns

Index(['ADM0_NAME', 'flow_name', 'cf', 'cf_median', 'cf_std'], dtype='object')